# Alert 생성기 실험 ④ — 정답 · 생성기 · 단순 연결을 그림으로 비교

세탁 시도 하나를 골라 세 가지를 나란히 그려 보는 HTML 을 만든다.

| 그림 | 내용 |
|---|---|
| ① 정답 | 그 세탁 시도의 거래 (Patterns 파일의 BEGIN ~ END 한 덩어리) |
| ② 생성기 | 그 시도의 거래가 들어간 Alert 전부 (01번 결과) |
| ③ 단순 연결 | 그 시도의 씨앗이 들어간 묶음 전부 (씨앗끼리 계좌를 공유하면 같은 묶음) |

- 자금세탁 거래는 빨간색, 정상 거래는 파란색이다. 굵은 선이 이 시도의 거래다
- 그림 방식은 `10_missed_patterns` 화면과 같다: **흐름 순서**(왼쪽 → 오른쪽 = 돈이 흐르는 순서, 기본), 자유 배치, 시간축 가로, 시간축 세로
- 세 그림은 같은 거래에 같은 번호, 같은 계좌에 같은 글자(A, B, C…)를 쓴다. 번호는 아래 표의 `#` 이다
- 생성기·단순 연결 그림에는 정답인데 묶음에 빠진 거래가 점선(✕)으로 남는다. 무엇이 빠지고 무엇이 더 붙었는지 바로 보인다
- 01번을 먼저 돌려야 한다. 생성기를 다시 돌리지는 않는다
- **02번과 다른 점**: 여기서는 STACK · BIPARTITE 를 빼지 않고 실제로 만들어진 묶음을 그대로 그린다. 그래서 건수가 02번 표와 조금 다를 수 있다

## 0. 설정

In [1]:
DATASET = 'HI-Medium'
RUN = 'before_after_grouping'      # 01번 결과 폴더
BASELINE_HOURS = None              # 단순 연결의 시간 제한(시간). None = 계좌만 공유하면 묶는다 (02번의 B0)
MIN_IN_WINDOW = 2                  # 평가 구간 안 거래가 이 건수 이상인 시도만 목록에 넣는다

In [2]:
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd
import gen_lib as GL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
RUN_DIR = Path.cwd() / 'results' / DATASET / RUN
OUT = Path.cwd() / 'results' / DATASET / 'compare_graphs'
OUT.mkdir(parents=True, exist_ok=True)
TEMPLATE = Path.cwd() / 'compare_graphs_template.html'
assert TEMPLATE.is_file(), f'{TEMPLATE} 이 없다'
run = json.load(open(RUN_DIR / 'run.json', encoding='utf-8'))
a, b = run['test_window']
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
con = duckdb.connect()
print('01번 실행:', run['scores'], '| 기준', run['threshold'], '| 평가 구간', a, '~', b)

01번 실행: ../../parquets/HI-Medium/service_scores/multi280_hgb36_v1_test_scores.parquet | 기준 0.194997844 | 평가 구간 2022-09-08 ~ 2022-09-16


## 1. 01번 결과 + 단순 연결
`members.parquet` 의 씨앗(모델 씨앗 + 보강)을 계좌 공유로 잇는다. 02번의 단순 연결과 같은 함수다. 거래 2건 이상인 묶음만 묶음으로 친다.

In [3]:
members = pd.read_parquet(RUN_DIR / 'members.parquet')
con.register('member_ids', members[['edge_id']])
where = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst FROM read_parquet('{BASE / 'transactions.parquet'}') t JOIN member_ids USING (edge_id)""").df()
members = members.merge(where, on='edge_id', how='left')
assert members.ts.notna().all(), '거래 파일에 없는 edge_id 가 있다'
is_seed = members.seed_model | members.promoted
seeds = members[is_seed].sort_values(['ts', 'edge_id']).reset_index(drop=True)
seeds['component'] = GL.link_by_account(seeds, BASELINE_HOURS)
size = seeds.groupby('component').edge_id.transform('size')
component = dict(zip(seeds.edge_id[size >= 2], pd.factorize(seeds.component[size >= 2])[0] + 1))      # 거래 → 묶음 번호 (1부터)
print(f'씨앗 {len(seeds):,} | 단순 연결 묶음 {len(set(component.values())):,} 개 (혼자 남은 씨앗 {int((size < 2).sum()):,}) | Alert {members.alert[members.role > 0].nunique():,} 개')

씨앗 14,219 | 단순 연결 묶음 1,729 개 (혼자 남은 씨앗 1,891) | Alert 2,051 개


## 2. 정답: 세탁 시도
한 거래가 두 시도에 들어 있으면 번호가 작은 시도로 둔다(02번과 같다). 목록에는 평가 구간 안 거래가 `MIN_IN_WINDOW` 건 이상인 시도를 넣는다.

In [4]:
PAT = BASE / 'patterns.parquet'
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
con.register('attempt_of', attempt_of)
# 시도의 모든 거래 (평가 구간 밖 거래도 그림에 점선으로 넣는다)
rows = con.execute(f"""SELECT a.attempt_id, a.pattern_type, t.edge_id, t.ts
                       FROM attempt_of a JOIN read_parquet('{BASE / 'transactions.parquet'}') t USING (edge_id)""").df()
rows['win'] = (rows.ts >= start) & (rows.ts < end)
info = rows.groupby('attempt_id').agg(pattern_type=('pattern_type', 'first'), n=('edge_id', 'size'), nin=('win', 'sum'), last_ts=('ts', 'max'))
info['after'] = info.last_ts >= end                          # 평가 구간 뒤로 이어지는 시도
listed = info[info.nin >= MIN_IN_WINDOW]
print(f'평가 구간에 걸친 시도 {int((info.nin >= 1).sum()):,} 개 | 목록에 넣는 시도 {len(listed):,} 개')
listed.groupby('pattern_type').agg(시도=('n', 'size'), 구간_뒤로_이어짐=('after', 'sum'), 구간_안_거래_중앙값=('nin', 'median'))

평가 구간에 걸친 시도 2,189 개 | 목록에 넣는 시도 1,670 개


,시도,구간_뒤로_이어짐,구간_안_거래_중앙값
pattern_type,,,
BIPARTITE,151,20,7.0
CYCLE,236,69,4.0
FAN-IN,209,70,5.0
FAN-OUT,183,57,5.0
GATHER-SCATTER,267,145,8.0
RANDOM,168,45,4.0
SCATTER-GATHER,228,65,7.0
STACK,228,79,7.0


## 3. 그림에 쓸 거래 표
01번의 씨앗·Alert 구성 거래와, 목록에 든 시도의 거래를 합친다. 계좌·시각·금액·점수를 붙인다.

In [5]:
ids = pd.DataFrame({'edge_id': pd.unique(np.concatenate([members.edge_id.to_numpy(), rows.edge_id[rows.attempt_id.isin(listed.index)].to_numpy()]))})
con.register('ids', ids)
tx = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst, t.paid, t.payment_currency, CAST(t.is_laundering AS INTEGER) AS y, s.score
                     FROM ids JOIN read_parquet('{BASE / 'transactions.parquet'}') t USING (edge_id)
                     LEFT JOIN read_parquet('{run['scores']}') s USING (edge_id)
                     ORDER BY t.ts, t.edge_id""").df()
assert len(tx) == len(ids)
tx['usd'] = GL.to_usd(tx.paid, tx.payment_currency)
tx = tx.merge(attempt_of, on='edge_id', how='left').merge(members[['edge_id', 'seed_model', 'promoted', 'role', 'alert', 'reason']], on='edge_id', how='left')
tx['win'] = (tx.ts >= start) & (tx.ts < end)
tx['seed'] = np.select([tx.seed_model.eq(True), tx.promoted.eq(True)], [1, 2], 0)      # 1 = 모델 씨앗, 2 = 보강으로 들어온 씨앗, 0 = 씨앗 아님
tx['role'] = tx.role.fillna(0).astype(int)
tx['al'] = np.where(tx.role > 0, tx.alert.fillna(-1), -1).astype(int)
tx['cp'] = tx.edge_id.map(component).fillna(-1).astype(int)
tx['att'] = tx.attempt_id.fillna(-1).astype(int)
print(f'거래 {len(tx):,} 건 | 세탁 {int(tx.y.sum()):,} | 씨앗 {int((tx.seed > 0).sum()):,} | Alert 구성 {int((tx.al >= 0).sum()):,} | 단순 연결 묶음에 든 거래 {int((tx.cp >= 0).sum()):,}')

거래 20,591 건 | 세탁 18,772 | 씨앗 14,219 | Alert 구성 11,526 | 단순 연결 묶음에 든 거래 12,328


## 4. 시도별 결과
평가 구간 안 거래 기준이다. `포함률` = 그 시도의 거래를 가장 많이 담은 묶음 하나에 들어간 비율.

| 분류 | 뜻 |
|---|---|
| 온전 | 전부 한 묶음에 들어감 (포함률 1) |
| 쪼개짐 | 두 묶음 이상에 나뉘어 들어감 |
| 일부만 | 한 묶음에 일부만 들어감 |
| 안 묶임 | 어느 묶음에도 없음 |

In [6]:
def per_attempt(column):
    t = tx[tx.win & tx.att.isin(listed.index) & (tx[column] >= 0)]
    counts = t.groupby(['att', column]).size()
    return counts.groupby('att').max().reindex(listed.index, fill_value=0), counts.groupby('att').size().reindex(listed.index, fill_value=0)

def verdict(cov, groups):
    return np.select([groups == 0, cov >= 1, groups >= 2], ['안 묶임', '온전', '쪼개짐'], '일부만')

stat = listed[['pattern_type', 'n', 'nin', 'after']].copy()
for key, column in [('g', 'al'), ('b', 'cp')]:
    best, groups = per_attempt(column)
    stat[f'{key}_cov'], stat[f'{key}_groups'] = best / stat.nin, groups
    stat[f'{key}_결과'] = verdict(stat[f'{key}_cov'].to_numpy(), groups.to_numpy())
ORDER = ['온전', '쪼개짐', '일부만', '안 묶임']
pd.crosstab(stat.g_결과, stat.b_결과, margins=True).reindex(index=ORDER + ['All'], columns=ORDER + ['All'], fill_value=0).rename_axis(index='생성기', columns='단순 연결')

단순 연결,온전,쪼개짐,일부만,안 묶임,All
생성기,,,,,
온전,966,0,9,1,976
쪼개짐,151,128,11,0,290
일부만,92,22,99,0,213
안 묶임,42,11,42,96,191
All,1251,161,161,97,1670


In [7]:
# 패턴별: 생성기 결과
pd.crosstab(stat.pattern_type, stat.g_결과).reindex(columns=ORDER, fill_value=0)

g_결과,온전,쪼개짐,일부만,안 묶임
pattern_type,,,,
BIPARTITE,5,15,31,100
CYCLE,179,31,12,14
FAN-IN,158,13,30,8
FAN-OUT,151,13,17,2
GATHER-SCATTER,127,54,71,15
RANDOM,140,20,5,3
SCATTER-GATHER,183,16,22,7
STACK,33,128,25,42


## 5. HTML 쓰기
`compare_graphs_template.html` 에 데이터를 넣어 파일 하나로 만든다. 인터넷 없이 브라우저로 열면 된다.
왼쪽 목록에서 시도를 고르고, 위쪽에서 패턴·결과로 걸러 볼 수 있다. 거래 위에 마우스를 올리면 세 그림과 아래 표에서 같은 거래가 같이 강조된다.
화면이 좁으면 세 그림이 위아래로 놓인다(`나란히 / 위아래로 크게` 로 바꿀 수 있다).

In [8]:
patterns = sorted(set(attempt_of.pattern_type))
reasons = [''] + sorted(set(tx.reason.dropna()) - {''})
none = lambda s: [None if pd.isna(v) else v for v in s]
data = dict(
    meta=dict(dataset=DATASET, window=[a, b], model=Path(run['scores']).name.replace('_test_scores.parquet', ''), threshold=round(float(run['threshold']), 6),
              promotion='끔' if run['seed_low'] is None else f"{run['seed_low']} / {run['chain_floor']}",
              baseline='계좌 공유 (시간 제한 없음)' if BASELINE_HOURS is None else f'계좌 공유 + {BASELINE_HOURS}시간 이내',
              seeds=int((tx.seed > 0).sum()), alerts=int(tx.al[tx.al >= 0].nunique()), components=int(tx.cp[tx.cp >= 0].nunique())),
    patterns=patterns, reasons=reasons,
    tx=dict(id=tx.edge_id.tolist(), s=tx.src.tolist(), d=tx.dst.tolist(), t=(tx.ts.astype('datetime64[s]').astype('int64') // 60).tolist(),
            usd=none(tx.usd.round(2)), y=tx.y.tolist(), att=tx.att.tolist(), pt=tx.pattern_type.map({p: i for i, p in enumerate(patterns)}).fillna(-1).astype(int).tolist(),
            sc=none(tx.score.round(4)), seed=tx.seed.tolist(), role=tx.role.tolist(), al=tx.al.tolist(), cp=tx.cp.tolist(),
            rs=tx.reason.fillna('').map({r: i for i, r in enumerate(reasons)}).fillna(0).astype(int).tolist(), win=tx.win.astype(int).tolist()),
    attempts=[dict(id=int(i), p=patterns.index(r.pattern_type), n=int(r.n), nin=int(r.nin), after=int(r.after),
                   g=[round(float(r.g_cov), 4), int(r.g_groups)], b=[round(float(r.b_cov), 4), int(r.b_groups)]) for i, r in stat.iterrows()])
html = TEMPLATE.read_text(encoding='utf-8')
assert html.count('/*__DATA__*/null') == 1, '템플릿에 데이터 자리가 없다'
html = html.replace('/*__DATA__*/null', json.dumps(data, ensure_ascii=False, separators=(',', ':'), allow_nan=False).replace('</', '<\\/'))
path = OUT / 'compare_graphs.html'
path.write_text(html, encoding='utf-8')
stat.to_csv(OUT / 'attempts.csv')
print(path, f'{path.stat().st_size / 1e6:.1f} MB | 시도 {len(stat):,} 개 | 거래 {len(tx):,} 건')

/home/donghyeon/workspace/money_laundry_ai_/code/rule_engine/results/HI-Medium/compare_graphs/compare_graphs.html 1.6 MB | 시도 1,670 개 | 거래 20,591 건
